# Day 05：梯度消失 —— 深度网络的「老年痴呆」

> 👁️ 第三周 · 视觉的征服 · 第 5 天

昨天我们用 LeNet-5 在 MNIST 上取得了 98%+ 的准确率。LeNet-5 只有 5 层，训练很顺利。但如果我们把网络加深到 20 层、50 层呢？

你会发现一个诡异的现象：**网络越深，效果反而越差**。不是过拟合——训练集上的准确率也在下降。更诡异的是，**靠近输入的层几乎不更新**，它们的梯度接近于零。

这就是**梯度消失（Vanishing Gradient）**——深度学习历史上最大的绊脚石之一。它让深层网络的训练几乎不可能，直到 ReLU 激活函数的出现才被缓解。

**今天的任务**：理解梯度消失的根本原因，亲眼看到 Sigmoid 的梯度如何逐层衰减，以及 ReLU 如何解决这个问题。

---
## 1. 历史背景：深度网络的「黑暗时代」

### 为什么深度网络训练不了？

在 2006 年之前，训练超过 3-4 层的神经网络几乎是不可能的。不是算力不够——是**梯度消失了**。

想象你在训练一个 10 层的全连接网络。反向传播时，梯度从第 10 层传回第 1 层，需要经过 9 次链式法则的乘法。如果每次乘法都把梯度缩小一点（比如 ×0.25），那么：

```
第 10 层的梯度: 1.0
第 9 层的梯度:  1.0 × 0.25 = 0.25
第 8 层的梯度:  0.25 × 0.25 = 0.0625
第 7 层的梯度:  0.0625 × 0.25 = 0.0156
...
第 1 层的梯度:  1.0 × 0.25^9 ≈ 0.000004
```

第 1 层的梯度只有第 10 层的 **25 万分之一**！这意味着第 1 层几乎不学习——它的权重几乎不变，等于白搭。

### Sigmoid 是罪魁祸首

早期神经网络的标准激活函数是 **Sigmoid**：

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

Sigmoid 的导数：

$$\sigma'(x) = \sigma(x)(1 - \sigma(x))$$

Sigmoid 导数的最大值是 0.25（在 x=0 处），在两端趋近于 0。这意味着每次经过 Sigmoid，梯度**最多**被乘以 0.25，通常更小。

10 层 Sigmoid 网络 → 梯度衰减至少 $0.25^{10} \approx 10^{-6}$ 倍。这就是梯度消失的数学根源。

### 2006 年的突破：逐层预训练

2006 年，Geoffrey Hinton 提出了一个「曲线救国」的方案：**逐层预训练**。

思路是：
1. 先训练第一层（作为自编码器），让它学会提取基础特征
2. 固定第一层，训练第二层
3. 固定前两层，训练第三层
4. ……
5. 最后用反向传播微调整个网络

这个方法绕过了梯度消失——每层单独训练，不需要梯度从顶层传到底层。但它很麻烦，而且不是端到端的。

### 2011 年：ReLU 的救赎

真正的解决方案来自一个看似简单的激活函数：**ReLU（Rectified Linear Unit）**。

$$\text{ReLU}(x) = \max(0, x)$$

ReLU 的导数极其简单：
- 当 x > 0 时，导数为 **1**
- 当 x ≤ 0 时，导数为 **0**

导数为 1 意味着：**梯度在正向区域完全不衰减**！10 层 ReLU 网络 → 梯度衰减 $1^{10} = 1$ 倍（前提是每层的输入都 > 0）。

2011 年，Xavier Glorot 和 Yoshua Bengio 在论文中证明了 ReLU 能有效缓解梯度消失。2012 年的 AlexNet 使用了 ReLU，大获成功。从此，ReLU 成为了深度网络的标配激活函数。

### 梯度消失的「帮凶」

除了 Sigmoid 的导数 < 1，还有几个因素会加剧梯度消失：

**权重初始化不当**：如果初始权重太小，每层的输出方差会逐层缩小 → 梯度也逐层缩小。

**深层网络的链式法则**：反向传播的梯度是各层导数（雅可比矩阵）的连乘积。如果每层的雅可比矩阵的「最大奇异值」< 1，梯度就会指数级衰减。

$$\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial h_L} \cdot \frac{\partial h_L}{\partial h_{L-1}} \cdot ... \cdot \frac{\partial h_2}{\partial h_1} \cdot \frac{\partial h_1}{\partial W_1}$$

如果每个 $\frac{\partial h_i}{\partial h_{i-1}}$ 的模都 < 1，连乘后梯度趋近于 0。

---
## 2. 生活隐喻：传话游戏

### 隐喻一：传话游戏

梯度消失最经典的生活类比是**传话游戏**：

10 个人排成一排，第一个人看到一句话（「今天天气真好」），悄悄告诉第二个人，第二个人告诉第三个人……传到第十个人时，他说出来的可能是「鸡腿真好吃」——信息在传递过程中逐渐失真了。

在神经网络中：
- 第一个人 = 输入层
- 最后一个人 = 输出层
- 传话过程 = 反向传播
- 信息失真 = 梯度衰减

如果每个人传话时都「模糊」一点（×0.9），传 10 次后信息只剩 $0.9^{10} \approx 35\%$。传 50 次后只剩 $0.9^{50} \approx 0.5\%$——几乎什么都没剩下。

Sigmoid 的传话效率更差——每次最多 ×0.25，10 次后只剩 $0.25^{10} \approx 0.0001\%$。

### 隐喻二：老年痴呆

梯度消失就像神经网络的「老年痴呆」：

- **浅层网络**（年轻人）：记忆力好，能记住最近发生的事情（梯度正常传播）
- **深层网络**（老年人）：记不住最近发生的事情，越久远的事情越模糊（梯度消失）

具体来说：
- 输出层（最近发生的事情）：记得很清楚，梯度正常
- 中间层（几年前的事情）：有点模糊，梯度很小
- 输入层（几十年前的事情）：完全想不起来，梯度几乎为零

ReLU 就像一种「记忆增强药」——它让信息传递时不会衰减（导数为 1），所以即使是 100 层的网络，输入层也能「记住」输出层的反馈。

### 隐喻三：水管系统

把梯度想象成水管中的水压：

- **Sigmoid 网络**：水管上每隔一段就有一个「减压阀」，水压逐段降低。到水管末端（输入层），水压几乎为零——水龙头打开也没水出来。
- **ReLU 网络**：水管上没有减压阀（导数为 1），水压从头到尾保持一致。唯一的代价是有些水管段可能被「堵住」（ReLU 的负半区导数为 0）。

这就是为什么 ReLU 能让深层网络训练起来——它保持了梯度的「水压」。

---
## 3. 数学直觉：梯度如何逐层衰减？

### Sigmoid 的导数分析

Sigmoid 函数及其导数：

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

$$\sigma'(x) = \sigma(x)(1 - \sigma(x))$$

关键性质：
- $\sigma'(x)$ 的最大值是 **0.25**（在 x=0 处）
- 当 |x| > 4 时，$\sigma'(x) < 0.018$（几乎为零）
- 当 |x| > 6 时，$\sigma'(x) < 0.0025$

在深层网络中，经过几层后，很多神经元的输入会落入 Sigmoid 的「饱和区」（|x| 很大），导数接近于零。这就是梯度消失的「加速器」。

### ReLU 的导数分析

$$\text{ReLU}(x) = \max(0, x)$$

$$\text{ReLU}'(x) = \begin{cases} 1 & x > 0 \\ 0 & x \leq 0 \end{cases}$$

关键性质：
- 当 x > 0 时，导数为 **1**（梯度完全不衰减）
- 当 x ≤ 0 时，导数为 **0**（神经元「死亡」，梯度为零）

ReLU 的缺点是「死亡 ReLU」问题——如果某个神经元的所有输入都 ≤ 0，它的梯度永远为零，再也不会更新。Leaky ReLU（$\max(0.01x, x)$）通过给负半区一个很小的斜率来缓解这个问题。

### 梯度衰减的定量分析

假设一个 L 层的全连接网络，每层使用 Sigmoid 激活：

$$h_l = \sigma(W_l h_{l-1} + b_l)$$

损失对第一层权重的梯度：

$$\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial h_L} \cdot \prod_{l=L}^{2} \frac{\partial h_l}{\partial h_{l-1}} \cdot \frac{\partial h_1}{\partial W_1}$$

其中：

$$\frac{\partial h_l}{\partial h_{l-1}} = \text{diag}(\sigma'(W_l h_{l-1} + b_l)) \cdot W_l$$

如果 $\|\text{diag}(\sigma'(\cdot)) \cdot W_l\| < 1$ 对所有层成立，那么：

$$\left\|\frac{\partial L}{\partial W_1}\right\| \leq \left\|\frac{\partial L}{\partial h_L}\right\| \cdot \prod_{l=L}^{2} \|\text{diag}(\sigma'(\cdot)) \cdot W_l\|$$

梯度随层数**指数衰减**。这就是为什么在 ReLU 出现之前，训练超过 5 层的网络几乎不可能。

---
## 4. 代码实现：亲眼看到梯度消失

下面用代码直观展示 Sigmoid 和 ReLU 在深层网络中的梯度传播差异。

In [ ]:
# 设置中文字体
import matplotlib.pyplot as plt
try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# 创建两个深层网络：一个用 Sigmoid，一个用 ReLU
class DeepSigmoidNet(nn.Module):
    def __init__(self, num_layers=10):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(10, 10) for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = torch.sigmoid(layer(x))
        return x

class DeepReLUNet(nn.Module):
    def __init__(self, num_layers=10):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(10, 10) for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = torch.relu(layer(x))
        return x

# 创建两个网络
sigmoid_net = DeepSigmoidNet(10)
relu_net = DeepReLUNet(10)

# 用相同的随机输入
x = torch.randn(1, 10)

# 前向传播
y_sigmoid = sigmoid_net(x)
y_relu = relu_net(x)

# 反向传播，记录每层权重的梯度范数
loss_sigmoid = y_sigmoid.sum()
loss_relu = y_relu.sum()

loss_sigmoid.backward()
sigmoid_grads = [layer.weight.grad.norm().item()
                 for layer in sigmoid_net.layers]

loss_relu.backward()
relu_grads = [layer.weight.grad.norm().item()
              for layer in relu_net.layers]

# 可视化
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

layers = range(1, 11)
ax.semilogy(layers, sigmoid_grads, 'r-o', label='Sigmoid 网络', linewidth=2)
ax.semilogy(layers, relu_grads, 'b-o', label='ReLU 网络', linewidth=2)
ax.set_xlabel('层编号（1=靠近输入, 10=靠近输出）')
ax.set_ylabel('梯度范数（对数尺度）')
ax.set_title('梯度消失对比：Sigmoid vs ReLU（10层网络）')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Week03/images/cnn_day05_gradient_vanishing.png', dpi=150, bbox_inches='tight')
plt.show()

print("观察要点：")
print("  Sigmoid 网络：梯度从第10层到第1层急剧衰减（对数尺度下降）")
print("  ReLU 网络：梯度在各层之间保持相对稳定")
print("  这就是为什么深层网络必须用 ReLU！")

### 不同深度的梯度衰减

对比不同层数下 Sigmoid 网络的梯度衰减程度。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

def measure_gradient_decay(num_layers, activation='sigmoid'):
    """测量不同层数下第一层权重的梯度范数"""
    layers = nn.ModuleList([nn.Linear(10, 10) for _ in range(num_layers)])
    x = torch.randn(1, 10)

    # 前向传播
    h = x
    for layer in layers:
        if activation == 'sigmoid':
            h = torch.sigmoid(layer(h))
        else:
            h = torch.relu(layer(h))

    # 反向传播
    loss = h.sum()
    loss.backward()

    # 返回第一层和最后一层的梯度范数
    first_grad = layers[0].weight.grad.norm().item()
    last_grad = layers[-1].weight.grad.norm().item()
    return first_grad, last_grad

# 测试不同层数
depths = [2, 5, 10, 15, 20]
sigmoid_first = []
sigmoid_last = []
relu_first = []
relu_last = []

for d in depths:
    fg, lg = measure_gradient_decay(d, 'sigmoid')
    sigmoid_first.append(fg)
    sigmoid_last.append(lg)

    fg, lg = measure_gradient_decay(d, 'relu')
    relu_first.append(fg)
    relu_last.append(lg)

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(depths, sigmoid_first, 'r-o', label='第一层梯度', linewidth=2)
ax1.semilogy(depths, sigmoid_last, 'r--o', label='最后一层梯度', linewidth=2)
ax1.set_xlabel('网络层数')
ax1.set_ylabel('梯度范数（对数尺度）')
ax1.set_title('Sigmoid 网络：梯度随深度衰减')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.semilogy(depths, relu_first, 'b-o', label='第一层梯度', linewidth=2)
ax2.semilogy(depths, relu_last, 'b--o', label='最后一层梯度', linewidth=2)
ax2.set_xlabel('网络层数')
ax2.set_ylabel('梯度范数（对数尺度）')
ax2.set_title('ReLU 网络：梯度保持稳定')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('梯度消失 vs 网络深度', fontsize=14)
plt.tight_layout()
plt.savefig('Week03/images/cnn_day05_depth_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("关键发现：")
print("  Sigmoid 网络：层数越多，第一层梯度越小（指数衰减）")
print("  ReLU 网络：层数增加，第一层梯度基本不变")
print("  这就是为什么 ResNet 能做到 152 层——ReLU 保证了梯度能传到底层")

---
## 5. 验证实验：浅层 vs 深层 Sigmoid 网络

用一个实际的 MNIST 分类任务来验证：同样的 Sigmoid 激活，浅层网络和深层网络的性能差异。

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 加载 MNIST
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('Week03/data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('Week03/data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

class SigmoidMLP(nn.Module):
    """使用 Sigmoid 激活的 MLP"""
    def __init__(self, hidden_layers=1):
        super().__init__()
        layers = []
        in_dim = 28 * 28
        for _ in range(hidden_layers):
            layers.append(nn.Linear(in_dim, 256))
            layers.append(nn.Sigmoid())  # 使用 Sigmoid
            in_dim = 256
        layers.append(nn.Linear(256, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        return self.net(x)

def train_and_eval(hidden_layers):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = SigmoidMLP(hidden_layers).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(5):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

print("Sigmoid 网络：不同深度的性能对比")
print("=" * 50)
for depth in [1, 2, 3, 5, 8]:
    acc = train_and_eval(depth)
    print(f"  {depth} 层 Sigmoid MLP: 准确率 = {acc:.2f}%")

print("\n结论：Sigmoid 网络超过 3 层后，性能不升反降")
print("  不是模型容量不够——是梯度消失了，前面的层没学到东西")

### ReLU 的对比

同样的实验，但使用 ReLU 激活函数。

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('Week03/data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('Week03/data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

class ReLUMLP(nn.Module):
    """使用 ReLU 激活的 MLP"""
    def __init__(self, hidden_layers=1):
        super().__init__()
        layers = []
        in_dim = 28 * 28
        for _ in range(hidden_layers):
            layers.append(nn.Linear(in_dim, 256))
            layers.append(nn.ReLU())  # 使用 ReLU
            in_dim = 256
        layers.append(nn.Linear(256, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        return self.net(x)

def train_and_eval(hidden_layers):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ReLUMLP(hidden_layers).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(5):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

print("ReLU 网络：不同深度的性能对比")
print("=" * 50)
for depth in [1, 2, 3, 5, 8]:
    acc = train_and_eval(depth)
    print(f"  {depth} 层 ReLU MLP: 准确率 = {acc:.2f}%")

print("\n结论：ReLU 网络越深，性能越好（或至少不下降）")
print("  梯度能正常传播到底层，每层都在有效学习")

---
## 翻译词典

| 生活直觉 | 深度学习术语 |
|----------|-------------|
| 传话游戏信息失真 | 梯度消失（Vanishing Gradient） |
| 老年痴呆（记不住早期的事） | 浅层梯度趋近于零 |
| 记忆增强药 | ReLU 激活函数 |
| 水管减压阀 | Sigmoid 导数 < 1 |
| 水管堵住 | ReLU 负半区导数为 0（死亡 ReLU） |
| 传话 10 次后只剩 35% | 梯度指数衰减 $0.9^{10}$ |
| 逐层预训练 | 绕开梯度消失的权宜之计 |

---
## 今日总结

| 概念 | 直觉理解 |
|------|----------|
| 梯度消失 | 反向传播时梯度逐层衰减，浅层几乎不更新 |
| Sigmoid | 导数最大 0.25，深层网络中梯度指数衰减 |
| ReLU | 正半区导数为 1，梯度不衰减 |
| 死亡 ReLU | 负半区导数为 0，神经元可能永久失活 |
| Leaky ReLU | 负半区给一个小斜率（如 0.01），避免死亡 |

**梯度消失的因果链**：
1. Sigmoid 导数 ≤ 0.25
2. 反向传播 = 链式法则连乘
3. 每层乘一个 < 1 的数 → 梯度指数衰减
4. 浅层梯度 ≈ 0 → 浅层不更新 → 深层网络 = 浅层网络

**ReLU 的解决方案**：
1. 正半区导数为 1 → 梯度不衰减
2. 负半区导数为 0 → 稀疏激活（也是正则化）
3. 计算简单（max(0,x)）→ 比 Sigmoid 快很多

**本周回顾**：
- Day 01：卷积运算——滑动窗口 + 逐元素相乘求和
- Day 02：卷积层——让卷积核自己学习特征
- Day 03：池化——压缩信息 + 平移不变性
- Day 04：LeNet-5——第一个成功商用的 CNN
- Day 05：梯度消失——ReLU 让深层网络成为可能

**下周预告**：下周我们将进入**训练技巧**的世界——Batch Normalization、Dropout、数据增强……这些技巧让训练深层网络从「可能」变成「稳定」。